# Lab 09: WGAN-GP on MNIST

**Goal:** Replace weight clipping with Gradient Penalty and see the improvement.

Changes from Lab 08 (WGAN):
- **Gradient Penalty** instead of weight clipping
- **InstanceNorm** instead of BatchNorm in Critic
- **Adam** instead of RMSProp (beta1=0, beta2=0.9)
- **No clipping** — Lipschitz is enforced through the loss

**Expected compute** (approximate): the gradient penalty needs a second backward pass, so this is the heaviest Day 2 lab. 25 epochs ≈ 20 min on Apple-silicon (`mps`), ≈ 10 min on a Colab T4, ≈ 25 min on CPU with the automatic 5k-image subset (~4.5 h on CPU with all 60k).

**How to use this lab:** answer each **Predict** prompt before running the cell.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 1. Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

mnist = datasets.MNIST("data", train=True, download=True, transform=transform)

# GP doubles the cost of every critic step. CPU-only machines use a 5k-image subset
# (None = all 60k images, used on GPU / mps).
TRAIN_SUBSET = 5_000 if device.type == "cpu" else None
if TRAIN_SUBSET:
    mnist = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET))
loader = DataLoader(mnist, batch_size=64, shuffle=True)

## 2. Critic & Generator

Key change: **InstanceNorm** in the Critic (not BatchNorm).  
BatchNorm mixes samples in a batch, which breaks the gradient penalty: GP constrains the gradient of *each* sample's score w.r.t. *that* sample, but with BatchNorm each score also depends on the other images in the batch.

In [ ]:
class Critic(nn.Module):
    """Scores images. InstanceNorm (not BatchNorm). No Sigmoid."""
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # 1 x 28 x 28
            nn.Conv2d(1, 64, 4, 2, 1),                 # -> 64 x 14 x 14
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),                # -> 128 x 7 x 7
            nn.InstanceNorm2d(128, affine=True),         # NOT BatchNorm!
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            # No Sigmoid
        )

    def forward(self, x):
        return self.model(x)


class Generator(nn.Module):
    """Takes noise, produces 28x28 image. BatchNorm is fine in G."""
    def __init__(self, z_dim=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.model(z)

In [ ]:
z_dim = 64

critic = Critic().to(device)
gen    = Generator(z_dim).to(device)

print(f"Critic params:    {sum(p.numel() for p in critic.parameters()):,}")
print(f"Generator params: {sum(p.numel() for p in gen.parameters()):,}")

## 3. The Gradient Penalty Function

This is the heart of WGAN-GP. Let's break it down:

1. **Interpolate** between real and fake images (random mix)
2. **Get Critic scores** on interpolated images
3. **Compute gradients** of scores w.r.t. the interpolated input
4. **Penalize** if gradient norm != 1

Implementation details that matter:
- `fake` is passed in already **detached** — the penalty trains the critic only.
- `create_graph=True` keeps the gradient computation in the graph, so `loss_C.backward()` can differentiate *through* it (a second derivative). Without it the penalty would have no effect on the weights.
- The norm is taken **per sample** over all pixels (`view(bs, -1).norm(2, dim=1)`), then averaged.

In [ ]:
def gradient_penalty(critic, real, fake, device):
    bs = real.size(0)

    # 1. Random interpolation between real and fake
    epsilon = torch.rand(bs, 1, 1, 1, device=device)   # one per image
    interpolated = (epsilon * real + (1 - epsilon) * fake).requires_grad_(True)

    # 2. Critic scores on interpolated images
    scores = critic(interpolated)

    # 3. Gradients of scores w.r.t. INPUT (not weights!)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,     # need to backprop through this
        retain_graph=True,
    )[0]

    # 4. Penalty: (||gradient||_2 - 1)^2
    gradients = gradients.view(bs, -1)
    penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    return penalty

print("Gradient penalty function ready.")

## 4. Training

Notice:
- **Adam** with beta1=0 (not RMSProp)
- **lambda_gp = 10** (standard value)
- **No weight clipping** — GP handles Lipschitz

**Predict:** what will the GP value be in epoch 0, and where will it settle?

In [ ]:
# Hyperparameters
lr        = 1e-4
n_critic  = 5
lambda_gp = 10       # gradient penalty weight
epochs    = 25

# Adam with beta1=0 (no momentum — more stable for GANs)
opt_C = torch.optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))
opt_G = torch.optim.Adam(gen.parameters(),    lr=lr, betas=(0.0, 0.9))

In [ ]:
# Track everything
history_C  = []
history_G  = []
history_GP = []    # track GP separately — watch it decrease!

fixed_noise = torch.randn(16, z_dim, device=device)

for epoch in range(epochs):
    ep_C, ep_G, ep_GP = [], [], []

    for real, _ in loader:
        real = real.to(device)
        bs = real.size(0)

        # ── Train Critic (5 times) ──
        for _ in range(n_critic):
            noise = torch.randn(bs, z_dim, device=device)
            fake  = gen(noise).detach()

            # Wasserstein loss + gradient penalty
            gp = gradient_penalty(critic, real, fake, device)
            loss_C = -critic(real).mean() + critic(fake).mean() + lambda_gp * gp

            opt_C.zero_grad()
            loss_C.backward()
            opt_C.step()
            # NO WEIGHT CLIPPING!

        # ── Train Generator (1 time) ──
        noise = torch.randn(bs, z_dim, device=device)
        fake  = gen(noise)
        loss_G = -critic(fake).mean()

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        ep_C.append(loss_C.item())
        ep_G.append(loss_G.item())
        ep_GP.append(gp.item())

    avg_C  = sum(ep_C) / len(ep_C)
    avg_G  = sum(ep_G) / len(ep_G)
    avg_GP = sum(ep_GP) / len(ep_GP)
    history_C.append(avg_C)
    history_G.append(avg_G)
    history_GP.append(avg_GP)

    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:3d} | C: {avg_C:+.4f} | G: {avg_G:+.4f} | GP: {avg_GP:.4f}")

## 5. Loss Curves + Gradient Penalty Curve

Three plots:
1. **Critic loss** — includes λ·GP; its Wasserstein part −(C(real) − C(fake)) is the W estimate with a minus sign, so as G improves the estimate shrinks and the curve rises toward 0 (same reading as Lab 08)
2. **Generator loss** — not comparable over time on its own (the critic keeps changing)
3. **Gradient Penalty** — usually spikes early, then settles at a small value (the L09 slide: ~0.5 → ~0.1 → ~0.01). If it stays high the critic is struggling with the constraint.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(history_C)
ax1.set_title("Critic Loss")
ax1.set_xlabel("Epoch")
ax1.grid(True, alpha=0.3)

ax2.plot(history_G, color="orange")
ax2.set_title("Generator Loss")
ax2.set_xlabel("Epoch")
ax2.grid(True, alpha=0.3)

ax3.plot(history_GP, color="green")
ax3.set_title("Gradient Penalty (should settle small)")
ax3.set_xlabel("Epoch")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Generated Images

In [ ]:
gen.eval()
with torch.no_grad():
    samples = gen(fixed_noise).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img = samples[i].squeeze()
    img = (img + 1) / 2
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle("WGAN-GP Generated Digits", fontsize=14)
plt.tight_layout()
plt.show()
gen.train();

## 7. Experiment: Compare Weight Distributions

Let's compare the Critic's weights in WGAN-GP vs the clipped WGAN from Lab 08.

**Predict:** what will this histogram look like compared with Lab 08's spikes at ±0.01?

In [ ]:
all_weights = []
for p in critic.parameters():
    all_weights.extend(p.data.cpu().flatten().tolist())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_weights, bins=100, edgecolor="black", alpha=0.7, color="steelblue")
ax.set_xlabel("Weight value")
ax.set_ylabel("Count")
ax.set_title("WGAN-GP Critic Weights (free to spread — no clipping!)")
plt.tight_layout()
plt.show()

import statistics
print(f"Weight range: [{min(all_weights):.4f}, {max(all_weights):.4f}]")
print(f"Std dev:      {statistics.stdev(all_weights):.4f}")
print("\nCompare to WGAN (Lab 08): weights were crushed to [-0.01, +0.01]")
print("Here they spread freely — the Critic uses its full capacity!")

## 8. Experiment: What Does the Interpolated Image Look Like?

The gradient penalty uses interpolated images between real and fake.  
Let's see what these look like.

In [ ]:
# Get one real and one fake image
real_batch, _ = next(iter(loader))
real_img = real_batch[0:1].to(device)   # one real image

gen.eval()
with torch.no_grad():
    fake_img = gen(torch.randn(1, z_dim, device=device))  # one fake image
gen.train()

# Show interpolation from fake to real
epsilons = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
fig, axes = plt.subplots(1, len(epsilons), figsize=(12, 2.5))

for i, eps in enumerate(epsilons):
    interp = eps * real_img + (1 - eps) * fake_img
    img = interp.cpu().squeeze()
    img = (img + 1) / 2
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"eps={eps}")
    axes[i].axis("off")

axes[0].set_title("eps=0\n(fake)", fontsize=10)
axes[-1].set_title("eps=1\n(real)", fontsize=10)
plt.suptitle("Interpolated Images (where we measure the gradient)", fontsize=12)
plt.tight_layout()
plt.show()

print("The gradient penalty checks the Critic's gradient at random points")
print("along this line between real and fake.")

### Your turn: λ ablation

Change `lambda_gp` to **0** (no constraint), **1** and **100**; re-run the model cell, the hyper-parameter cell and training with `epochs = 3`.

**Predict first**, then compare the GP curve and the samples. *Expected:* λ = 0 behaves like the un-clipped critic in Lab 08 (scores blow up); λ = 1 keeps a loose constraint (GP stays larger); λ = 100 over-constrains the critic and slows learning; λ = 10 is the paper's default for a reason.

## Key Takeaways

1. **Gradient Penalty** directly enforces Lipschitz — no more weight clipping
2. **InstanceNorm** in Critic — BatchNorm mixes samples, breaks GP
3. **Adam** with beta1=0 works great (RMSProp no longer needed)
4. **GP should settle small** during training (constraint getting satisfied)
5. **Weights spread freely** — Critic has full capacity, better results
6. **lambda=10** — standard value, rarely needs tuning

**Next: Lab 10 (Conditional GAN)** — tell the GAN WHAT to generate!